# 08 · MLP baseline (final phase)
**Owner:** shared by all members

64 × 64 grayscale (from the cropped slice), flattened; Dense 512 → 128 → 2 with dropout.

Final protocol: brain-cropped images, leak-free split (07), same metrics. Steps: (1) grid search on the validation split with seed 42, (2) the chosen configuration trained with 3 seeds, scored once on the test split, (3) learning curves.

In [1]:
%matplotlib inline
import os, sys
os.environ.setdefault("KERAS_BACKEND", "torch")   # Keras 3 backend used for our runs (CPU); "tensorflow" also works
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))        # make the project's src/ package importable

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import keras
from IPython.display import Image, display
from src import config, data_pipeline as dp, models, features, training

keras.utils.set_random_seed(config.SEED)
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
from src import experiments as E, evaluation as ev, finetune as F, gradcam as G
RERUN = False    # True = retrain everything here; False = load the saved runs produced by the same code
                 # (scripts/run_heads.py, scripts/run_finetune.py), which take hours on a CPU

Keras 3.15.1 | backend: torch


## Inputs

In [2]:
m = E.manifest_for("final")
X = E.mlp_arrays("final")
print(X.shape, "| class weights", E.class_weight(m))

(7013, 64, 64, 1) | class weights {0: 2.0855899419729207, 1: 0.6576700213479719}


## 1. Hyper-parameter search (validation split only)

In [3]:
grid_csv = config.RESULTS_DIR / "final" / "grid_mlp.csv"
if RERUN:
    cfg, grid = E.grid_search("mlp", X, m, E.MLP_GRID)
else:
    grid = pd.read_csv(grid_csv)
    cfg = json.loads((config.RESULTS_DIR / "final" / "chosen_mlp.json").read_text())["mlp"]
print("chosen:", cfg)
grid

chosen: {'lr': 0.0003, 'dropout': [0.5, 0.3]}


,lr,dropout,val_balanced_accuracy,val_auc,val_loss,best_epoch
0,0.0003,"(0.5, 0.3)",0.8860,0.9569,0.236472,17
1,0.0003,"(0.3, 0.2)",0.8841,0.9530,0.238779,13
2,0.0001,"(0.5, 0.3)",0.8751,0.9444,0.247903,13
3,0.0001,"(0.3, 0.2)",0.8690,0.9443,0.242399,10


## 2. Chosen configuration, 3 seeds (trained)

In [4]:
variant = "trained"
recs = E.run_seeds("mlp", variant, "final", X, m, cfg) if RERUN else E.load_runs("final", "mlp", variant)
pd.DataFrame({r["seed"]: {**{f"val_{k}": r["val_metrics"][k] for k in ["balanced_accuracy", "roc_auc"]},
                          **{f"test_{k}": r["test_metrics"][k] for k in ["accuracy", "recall", "specificity", "f1", "roc_auc"]},
                          "best_epoch": r["best_epoch"], "trainable_params": r["trainable_params"]} for r in recs}).T

,val_balanced_accuracy,val_roc_auc,test_accuracy,test_recall,test_specificity,test_f1,test_roc_auc,best_epoch,trainable_params
2026,0.8545,0.9148,0.9272,0.9618,0.8175,0.9525,0.9438,6.0,2163586.0
42,0.8860,0.9569,0.9457,0.9472,0.9409,0.9636,0.9802,17.0,2163586.0
7,0.8938,0.9685,0.9500,0.9602,0.9177,0.9669,0.9844,23.0,2163586.0


In [5]:
summary = E.summarise(recs)
pd.DataFrame(summary).drop(columns="n_seeds").T.round(4)

,mean,std
accuracy,0.9410,0.0121
precision,0.9659,0.0198
recall,0.9564,0.0080
specificity,0.8920,0.0656
f1,0.9610,0.0075
balanced_accuracy,0.9242,0.0300
roc_auc,0.9695,0.0223
FN,53.6667,9.8658
FP,42.0000,25.5147
